### 1. Завантаження даних VHI
**Завдання:** Для кожної з адміністративних одиниць України завантажити (urllib) файли з VHI-індексом. Додати дату та час до імені, передбачити уникнення повторного завантаження.

In [2]:
import urllib.request
import os
from datetime import datetime

def download_vhi(province_id):
    time_now = datetime.now().strftime("%Y-%m-%d_%H-%M")
    filename = f"vhi_data_{province_id}_{time_now}.csv"
    
    for file in os.listdir():
        if file.startswith(f"vhi_data_{province_id}_"):
            print(f"Область {province_id} вже завантажена.")
            return file 
            
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    
    response = urllib.request.urlopen(url)
    text = response.read().decode('utf-8')
    
    clean_text = text.replace("<tt><pre>", "").replace("</pre></tt>", "")
    
    with open(filename, 'w') as out_file:
        out_file.write(clean_text)
        
    print(f"Область {province_id} успішно завантажена та збережена!")
    return filename

all_files = []

for i in range(1, 28):
    file_name = download_vhi(i)
    all_files.append(file_name)

Область 1 успішно завантажена та збережена!
Область 2 успішно завантажена та збережена!
Область 3 успішно завантажена та збережена!
Область 4 успішно завантажена та збережена!
Область 5 успішно завантажена та збережена!
Область 6 успішно завантажена та збережена!
Область 7 успішно завантажена та збережена!
Область 8 успішно завантажена та збережена!
Область 9 успішно завантажена та збережена!
Область 10 успішно завантажена та збережена!
Область 11 успішно завантажена та збережена!
Область 12 успішно завантажена та збережена!
Область 13 успішно завантажена та збережена!
Область 14 успішно завантажена та збережена!
Область 15 успішно завантажена та збережена!
Область 16 успішно завантажена та збережена!
Область 17 успішно завантажена та збережена!
Область 18 успішно завантажена та збережена!
Область 19 успішно завантажена та збережена!
Область 20 успішно завантажена та збережена!
Область 21 успішно завантажена та збережена!
Область 22 успішно завантажена та збережена!
Область 23 успішно 

### 2. Зчитування, Data Cleaning та зміна індексів
**Завдання:** Зчитати файли у pandas dataframe. Прибрати зайві стовпці, пропуски. Змінити індекси областей на українську абетку (1 - Вінницька тощо).

In [ ]:
import pandas as pd

province_dict = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

def clean_and_merge_vhi(file_list):
    df_list = []
    
    for file in file_list:
        old_id = int(file.split('_')[2])
        new_id = province_dict[old_id]
        
        df = pd.read_csv(
            file,
            engine='python', 
            sep=',',
            skiprows=1,      
            names=['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty1', 'empty2'], 
            skipinitialspace=True
        )
        
        df = df[['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]
        
        df = df.replace({'<br>': ''}, regex=True)
        
        df = df[df['year'].astype(str).str.contains(r'^\d{4}$', na=False)]
        
        df = df.dropna()
        
        df['year'] = df['year'].astype(int)
        df['week'] = df['week'].astype(int)
        df['VHI'] = df['VHI'].astype(float)
        
        df = df[df['VHI'] != -1.00] 
            
        df['province_id'] = new_id
        df_list.append(df)
        
    return pd.concat(df_list, ignore_index=True)

df_clean = clean_and_merge_vhi(all_files)

display(df_clean.head())

,year,week,SMN,SMT,VCI,TCI,VHI,province_id
0,1982,1,0.053,260.31,45.01,39.46,42.23,22
1,1982,2,0.054,262.29,46.83,31.75,39.29,22
2,1982,3,0.055,263.82,48.13,27.24,37.68,22
3,1982,4,0.053,265.33,46.09,23.91,35.00,22
4,1982,5,0.050,265.66,41.46,26.65,34.06,22


### 3.1 Формування вибірки: VHI для області за вказаний рік
**Завдання:** Створити функцію, яка повертає ряд VHI для вказаної області (за українським індексом) та вказаного року.

In [6]:
def get_vhi_for_year(df, province_id, year):
    filtered_df = df[(df['province_id'] == province_id) & (df['year'] == year)]
    
    province_name = [k for k, v in province_dict.items() if v == province_id]
    print(f"Дані VHI для області {province_id} за {year} рік:")
    
    return filtered_df[['week', 'VHI']]

vhi_vinnytsia_2010 = get_vhi_for_year(df_clean, 1, 2010)

display(vhi_vinnytsia_2010.head())

Дані VHI для області 1 за 2010 рік:


,week,VHI
51684,1,52.04
51685,2,51.05
51686,3,52.37
51687,4,52.69
51688,5,53.16


### 3.2 Формування вибірки: VHI за діапазон років для вказаних областей
**Завдання:** Створити функцію, яка повертає дані VHI для заданого списку областей за вказаний діапазон років (наприклад, з 2015 по 2020 рік).

In [8]:
def get_vhi_for_range(df, province_ids, start_year, end_year):
    """
    Повертає дані VHI для масиву областей у вказаному діапазоні років.
    province_ids - список (наприклад, [1, 5, 12])
    """
    filtered_df = df[
        (df['year'] >= start_year) & 
        (df['year'] <= end_year) & 
        (df['province_id'].isin(province_ids))
    ]
    
    print(f"Дані VHI для областей {province_ids} за період з {start_year} по {end_year} рік:")
    
    return filtered_df[['year', 'week', 'province_id', 'VHI']]

vhi_range_data = get_vhi_for_range(df_clean, [1, 9], 2015, 2017)

display(vhi_range_data.sample(10))

Дані VHI для областей [1, 9] за період з 2015 по 2017 рік:


,year,week,province_id,VHI
51964,2015,21,1,45.14
51993,2015,50,1,46.63
52091,2017,44,1,47.78
51990,2015,47,1,34.57
23608,2016,31,9,54.17
23540,2015,15,9,39.79
23527,2015,2,9,47.79
52031,2016,36,1,32.39
23599,2016,22,9,55.92
51985,2015,42,1,35.18


### 3.3 Формування вибірки: Пошук екстремумів та статистики
**Завдання:** Знайти мінімальне, максимальне, середнє та медіанне значення VHI для вказаних областей та років.

In [9]:
def get_vhi_extremes(df, province_ids, year):
    filtered_df = df[(df['province_id'].isin(province_ids)) & (df['year'] == year)]
    
    stats_df = filtered_df.groupby('province_id')['VHI'].agg(
        Мінімум='min', 
        Максимум='max', 
        Середнє='mean', 
        Медіана='median'
    ).reset_index()
    
    print(f"Статистика VHI за {year} рік для областей: {province_ids}")
    return stats_df

vhi_stats = get_vhi_extremes(df_clean, [1, 9, 12], 2018)

display(vhi_stats)

Статистика VHI за 2018 рік для областей: [1, 9, 12]


,province_id,Мінімум,Максимум,Середнє,Медіана
0,1,36.64,69.48,50.458077,48.700
1,9,40.35,63.62,48.226346,47.105
2,12,36.72,59.63,50.338846,50.230
